# Metrics analysis

Loads the per-combo CSV files produced by [`run_sweep.py --output-dir`](run_sweep.py) and
provides small helpers for statistical comparison of any single metric between a
pair of evaluations (combos).

**Setup** (run once in a venv):

```bash
python3 -m venv .venv
source .venv/bin/activate
pip3 install -r requirements.txt
python3 -m ipykernel install --user --name master-diploma --display-name 'master-diploma (.venv)'
jupyter notebook demo/report.ipynb
```

Each CSV is expected to contain one row per repeated run of a single combo,
with parameter columns plus `rc`, `time_s`, and the scraped metric columns
(`orders`, `conflicts`, `avg_wait`, `avg_reverse`, `avg_speed`, `sim_speed`).
Missing values are represented as the literal `-`.

In [ ]:
from __future__ import annotations

import math
from itertools import combinations
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

pd.set_option('display.float_format', lambda v: f'{v:.4g}')

# Columns considered numeric metrics in the sweep CSVs.
METRIC_COLUMNS = [
    'orders',
    'conflicts',
    'avg_wait',
    'avg_reverse',
    'avg_speed',
    'avg_order_time',
]

## Loading

In [ ]:
def load_run(csv_path: str | Path) -> pd.DataFrame:
    """Load a single per-combo CSV produced by run_sweep.py.

    Returns a DataFrame in which the known metric columns are numeric
    (with the literal '-' coerced to NaN). All other columns are left as is.
    """
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    for col in METRIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].replace('-', np.nan), errors='coerce')
    return df


def load_runs(directory: str | Path, pattern: str = '*.csv') -> dict[str, pd.DataFrame]:
    """Load every CSV from `directory` into a dict keyed by file stem.

    The stem is the filename without the .csv extension. By default
    run_sweep.py names CSVs after a sanitized combo string, e.g.
    `router=astar_agents=10.csv`, so the keys end up being human-readable
    combo labels.
    """
    directory = Path(directory)
    runs: dict[str, pd.DataFrame] = {}
    for path in sorted(directory.glob(pattern)):
        runs[path.stem] = load_run(path)
    return runs


def overview(runs: dict[str, pd.DataFrame], metric: str) -> pd.DataFrame:
    """Compact mean/std/n table across all loaded runs for a single metric."""
    rows = []
    for label, df in runs.items():
        if metric not in df.columns:
            continue
        series = df[metric].dropna()
        rows.append({
            'combo': label,
            'n': int(series.size),
            'mean': series.mean() if series.size else np.nan,
            'std': series.std(ddof=1) if series.size > 1 else np.nan,
            'min': series.min() if series.size else np.nan,
            'max': series.max() if series.size else np.nan,
        })
    return pd.DataFrame(rows).set_index('combo')

## Statistical comparison of conflict-resolution methods

Each combo (CSV) is an independent sample of one method: every row is a separate
simulation run with its own seed. Sample sizes are large (n ≥ 10³), and the metrics
of interest (`orders`, `avg_speed`, `avg_wait`) are not normally distributed:

- `orders` is an integer count,
- `avg_speed` / `avg_wait` are agent-averaged but still have heavy right tails caused by
  occasional congestion (see e.g. the 1.089 spike among ~0.17 values in `time_s`).

We therefore use a **minimal non-parametric toolkit** rather than t-tests/ANOVA:

1. **Mann–Whitney U-test** — statistical significance of the difference between two
   methods. Assumes only independence of observations and makes no normality
   assumption; robust to the heavy tails above.
2. **Cliff's delta** — non-parametric effect size in `[-1, 1]`, derived from the same
   ranks as the U statistic. At our sample sizes any non-zero difference becomes
   "statistically significant" (p → 0), so a *size* of effect is mandatory to judge
   practical relevance. Standard thresholds (Romano et al., 2006): `|δ| < 0.147`
   negligible, `< 0.33` small, `< 0.474` medium, `≥ 0.474` large.
3. **Holm correction** — when comparing k methods pairwise we run k·(k−1)/2 tests
   per metric; Holm controls the family-wise error rate without sacrificing power
   like plain Bonferroni would.

Parametric checks (Shapiro–Wilk, Levene, Welch's t-test, ANOVA, bootstrap CI) are
intentionally omitted: with non-normal data and large n they would either reject
trivially (assumption tests) or merely duplicate the U-test result.

In [ ]:
def cliffs_delta(a: np.ndarray, b: np.ndarray) -> float:
    """Non-parametric effect size in [-1, 1].

    delta = P(A > B) - P(A < B), estimated from the samples.
    """
    a = np.asarray(a)
    b = np.asarray(b)
    gt = int((a[:, None] > b[None, :]).sum())
    lt = int((a[:, None] < b[None, :]).sum())
    return (gt - lt) / (a.size * b.size)


def _delta_magnitude(delta: float) -> str:
    ad = abs(delta)
    if ad < 0.147:
        return 'negligible'
    if ad < 0.33:
        return 'small'
    if ad < 0.474:
        return 'medium'
    return 'large'


def compare_pair(runs: dict[str, pd.DataFrame],
                 combo_a: str, combo_b: str, metric: str) -> pd.Series:
    """Mann-Whitney U + Cliff's delta for one (combo_a, combo_b, metric) triple."""
    a = runs[combo_a][metric].dropna().to_numpy()
    b = runs[combo_b][metric].dropna().to_numpy()
    u_stat, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    delta = cliffs_delta(a, b)
    return pd.Series({
        'metric': metric,
        'a': combo_a,
        'b': combo_b,
        'n_a': int(a.size),
        'n_b': int(b.size),
        'median_a': float(np.median(a)),
        'median_b': float(np.median(b)),
        'mwu_U': float(u_stat),
        'p_value': float(p),
        'cliffs_delta': float(delta),
        'effect': _delta_magnitude(delta),
    })


def compare_pairs(pairs: Iterable[tuple[str, str]],
                  runs: dict[str, pd.DataFrame],
                  metric: str,
                  alpha: float = 0.05) -> pd.DataFrame:
    """Run compare_pair over every pair in `pairs`.
    
    The resulting table includes Holm-adjusted p-values and a boolean
    `significant` flag at level `alpha`.
    """
    rows = [compare_pair(runs, a, b, metric) for a, b in pairs]
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    reject, p_adj, *_ = multipletests(out['p_value'].fillna(1.0),
                                      alpha=alpha, method='holm')
    out['p_holm'] = p_adj
    out['significant'] = reject
    return out.sort_values('p_holm').reset_index(drop=True)


def compare_all_pairs(runs: dict[str, pd.DataFrame],
                      metric: str,
                      alpha: float = 0.05) -> pd.DataFrame:
    """Run compare_pair over every unordered pair of combos for one metric.

    The resulting table includes Holm-adjusted p-values and a boolean
    `significant` flag at level `alpha`.
    """
    return compare_pairs(combinations(list(runs.keys()), 2), runs, metric, alpha)

### How to read the table

- `p_holm < 0.05` **and** `|cliffs_delta| ≥ 0.147` → methods differ both
  statistically and practically.
- `p_holm < 0.05` but `effect = negligible` → the difference is only an
  artefact of large n; ignore in practice.
- `p_holm ≥ 0.05` → no detectable difference for this metric.

In [30]:
# Example usage on the bundled sweep output.
runs = load_runs('reports')
overview(runs, 'avg_speed')

/tmp/ipykernel_22137/110497881.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = pd.to_numeric(df[col].replace('-', np.nan), errors='coerce')
/tmp/ipykernel_22137/110497881.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = pd.to_numeric(df[col].replace('-', np.nan), errors='coerce')


,n,mean,std,min,max
combo,,,,,
duration=86400_graph=maps_small_map.geojson_agents=100_router=astar_resolver=reverse,81,1.9,0.002433,1.894,1.905
duration=86400_graph=maps_small_map.geojson_agents=100_router=astar_resolver=semaphore,93,1.848,0.003951,1.838,1.857
duration=86400_graph=maps_small_map.geojson_agents=100_router=stat_resolver=reverse,80,1.974,0.001599,1.971,1.978
duration=86400_graph=maps_small_map.geojson_agents=100_router=stat_resolver=semaphore,87,1.894,0.002709,1.888,1.901


In [10]:
compare_pair(
    runs, 
    "duration=86400_graph=maps_small_map.geojson_agents=100_router=astar_resolver=semaphore", 
    "duration=86400_graph=maps_small_map.geojson_agents=100_router=stat_resolver=semaphore", 
    'orders')

metric                                                     orders
a               duration=86400_graph=maps_small_map.geojson_ag...
b               duration=86400_graph=maps_small_map.geojson_ag...
n_a                                                            93
n_b                                                            84
median_a                                                3.466e+04
median_b                                                3.485e+04
mwu_U                                                        1347
p_value                                                 5.644e-14
cliffs_delta                                              -0.6551
effect                                                      large
dtype: object

In [34]:
compare_all_pairs(runs, 'orders')

,metric,a,b,n_a,n_b,median_a,median_b,mwu_U,p_value,cliffs_delta,effect,p_holm,significant
0,orders,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,93,80,34737.0000,35641.0000,0.0000,0.0000,-1.0000,large,0.0000,True
1,orders,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,80,87,35641.0000,34841.0000,6960.0000,0.0000,1.0000,large,0.0000,True
2,orders,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,81,80,34526.0000,35641.0000,0.0000,0.0000,-1.0000,large,0.0000,True
3,orders,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,81,87,34526.0000,34841.0000,261.0000,0.0000,-0.9259,large,0.0000,True
4,orders,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,81,93,34526.0000,34737.0000,1055.5000,0.0000,-0.7198,large,0.0000,True
5,orders,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,93,87,34737.0000,34841.0000,2485.5000,0.0000,-0.3856,medium,0.0000,True


In [32]:
compare_all_pairs(runs, 'avg_speed')

,metric,a,b,n_a,n_b,median_a,median_b,mwu_U,p_value,cliffs_delta,effect,p_holm,significant
0,avg_speed,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,93,87,1.848,1.894,0,5.269e-31,-1,large,3.162e-30,True
1,avg_speed,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,81,93,1.9,1.848,7533,6.439e-30,1,large,3.22e-29,True
2,avg_speed,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,93,80,1.848,1.974,0,9.939e-30,-1,large,3.976e-29,True
3,avg_speed,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,80,87,1.974,1.894,6960,7.427e-29,1,large,2.228e-28,True
4,avg_speed,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,81,80,1.9,1.974,0,6.445e-28,-1,large,1.289e-27,True
5,avg_speed,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,81,87,1.9,1.894,6704,5.86e-24,0.9027,large,5.86e-24,True


In [28]:
compare_all_pairs(runs, 'avg_wait')

,metric,a,b,n_a,n_b,median_a,median_b,mwu_U,p_value,cliffs_delta,effect,p_holm,significant
0,avg_wait,duration=86400_graph=maps_small_map.geojson_ag...,duration=86400_graph=maps_small_map.geojson_ag...,93,84,43.9,49.83,2,1.93e-30,-0.9995,large,1.93e-30,True
